# Olist E-Commerce — Data Cleaning Report

**Prepared for:** Client (Olist Brazilian E-Commerce dataset)
**Prepared by:** Hafiz Gohar
**Source:** Kaggle — *Brazilian E-Commerce Public Dataset by Olist* (`olistbr/brazilian-ecommerce`), ~100k orders placed between Sept 2016 and Oct 2018 across 9 relational tables.

## The brief

You asked for a clean, trustworthy dataset that can feed a one-page sales dashboard (revenue trend, orders by month, top products, revenue by region, delivery performance). The raw Kaggle export cannot be used as-is: it contains duplicate rows, missing values that would silently distort averages, dates stored as text, and monetary columns with no currency or reconciliation against payments.

This notebook documents **every change made**, one problem per section, so you can audit the numbers on the dashboard back to the source.

| # | Problem | What was wrong | What I did |
|---|---------|----------------|------------|
| 1 | Duplicates | 261,831 exact duplicate rows in `geolocation`; 814 review IDs shared across multiple orders; 551 orders with more than one review | Dropped exact duplicates; kept one review per order (the most recently answered) |
| 2 | Nulls | 610 products with no category; 2 categories with no English name; 8 "delivered" orders with no delivery date; review titles/messages mostly empty | Filled categories with explicit `unknown`; added the 2 missing translations; flagged impossible delivery records; replaced empty comments with a `has_comment` flag |
| 3 | Date types | All 8 timestamp columns stored as plain text | Parsed to `datetime64`; derived `purchase_month`, `delivery_days`, `delivery_delay_days`, `delivered_on_time` |
| 4 | Currency | No currency unit on `price`/`freight_value`; 303 orders where payments ≠ item totals and 1 order with no payment at all | Renamed with `_brl` suffix, rounded to 2 dp, added `_usd` at a documented reference rate, flagged the 304 orders |

The output is a set of clean CSVs in `data/clean/` plus two dashboard-ready tables in `data/dashboard/`.

## 0. Setup and load

In [1]:
import pandas as pd
import numpy as np
import unicodedata
from pathlib import Path

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

RAW = Path("data/raw")
CLEAN = Path("data/clean"); CLEAN.mkdir(parents=True, exist_ok=True)
DASH = Path("data/dashboard"); DASH.mkdir(parents=True, exist_ok=True)

tables = {
    "orders":     "olist_orders_dataset.csv",
    "items":      "olist_order_items_dataset.csv",
    "payments":   "olist_order_payments_dataset.csv",
    "reviews":    "olist_order_reviews_dataset.csv",
    "customers":  "olist_customers_dataset.csv",
    "products":   "olist_products_dataset.csv",
    "sellers":    "olist_sellers_dataset.csv",
    "geolocation":"olist_geolocation_dataset.csv",
    "categories": "product_category_name_translation.csv",
}
raw = {name: pd.read_csv(RAW / f) for name, f in tables.items()}

# Keep an untouched copy of the row counts so we can report before/after at the end.
before_rows = {k: len(v) for k, v in raw.items()}
pd.DataFrame({"rows": before_rows, "columns": {k: v.shape[1] for k, v in raw.items()}})

,rows,columns
orders,99441,8
items,112650,7
payments,103886,5
reviews,99224,7
customers,99441,5
products,32951,9
sellers,3095,4
geolocation,1000163,5
categories,71,2


## 1. Problem: Duplicates

### What was wrong
Three different kinds of duplication, each needing a different fix:

1. **Exact duplicate rows** — `geolocation` has 1,000,163 rows but 261,831 of them are byte-for-byte copies of another row.
2. **One review attached to many orders** — 814 `review_id` values appear on 2+ different `order_id`s. A single customer review cannot belong to two orders; this is an export artefact that would double-count review scores.
3. **Many reviews on one order** — 551 orders have more than one review row. For a per-order dashboard we need exactly one.

Repeat customers (`customer_unique_id` appearing 3,345 times across different `customer_id`s) are **not** duplicates — one person placing several orders is real behaviour, so they are left alone.

In [2]:
dup_report = {}
for name, df in raw.items():
    dup_report[name] = {"rows": len(df), "exact_duplicate_rows": int(df.duplicated().sum())}
reviews = raw["reviews"]
dup_report["reviews"]["review_id_on_multiple_orders"] = int(reviews.review_id.duplicated().sum())
dup_report["reviews"]["orders_with_multiple_reviews"] = int(reviews.order_id.duplicated().sum())
pd.DataFrame(dup_report).T.fillna(0).astype(int)

,rows,exact_duplicate_rows,review_id_on_multiple_orders,orders_with_multiple_reviews
orders,99441,0,0,0
items,112650,0,0,0
payments,103886,0,0,0
reviews,99224,0,814,551
customers,99441,0,0,0
products,32951,0,0,0
sellers,3095,0,0,0
geolocation,1000163,261831,0,0
categories,71,0,0,0


### What I did

In [3]:
clean = {k: v.copy() for k, v in raw.items()}

# 1a. Exact duplicates: safe to drop everywhere (only geolocation actually has any).
for name in clean:
    clean[name] = clean[name].drop_duplicates().reset_index(drop=True)

# 1b/1c. Reviews: keep ONE review per order — the one answered most recently,
# which is the final state of the customer's feedback. Ties broken by creation date.
r = clean["reviews"]
r = r.sort_values(["order_id", "review_answer_timestamp", "review_creation_date"])
clean["reviews"] = r.drop_duplicates("order_id", keep="last").reset_index(drop=True)

print(f"geolocation: {before_rows['geolocation']:,} -> {len(clean['geolocation']):,} rows")
print(f"reviews:     {before_rows['reviews']:,} -> {len(clean['reviews']):,} rows "
      f"(one per order; {clean['reviews'].order_id.is_unique=})")

geolocation: 1,000,163 -> 738,332 rows
reviews:     99,224 -> 98,673 rows (one per order; clean['reviews'].order_id.is_unique=True)


### What the client would see
Review-based metrics (average score, % of 1-star orders) are now computed on exactly one review per order instead of being inflated by copied rows. The geolocation table shrinks by ~26% with zero information lost.

## 2. Problem: Nulls (missing values)

### What was wrong
Missing values are not uniformly "bad" — some are meaningful (an order that was cancelled has no delivery date), some are data-quality problems. I separated the two.

In [4]:
null_report = []
for name, df in clean.items():
    n = df.isna().sum()
    for col, cnt in n[n > 0].items():
        null_report.append({"table": name, "column": col, "nulls": int(cnt), "pct": round(100 * cnt / len(df), 2)})
pd.DataFrame(null_report).sort_values(["table", "nulls"], ascending=[True, False]).reset_index(drop=True)

,table,column,nulls,pct
0,orders,order_delivered_customer_date,2965,2.98
1,orders,order_delivered_carrier_date,1783,1.79
2,orders,order_approved_at,160,0.16
3,products,product_category_name,610,1.85
4,products,product_name_lenght,610,1.85
5,products,product_description_lenght,610,1.85
6,products,product_photos_qty,610,1.85
7,products,product_weight_g,2,0.01
8,products,product_length_cm,2,0.01
9,products,product_height_cm,2,0.01


Reading that table:

| Column(s) | Verdict | Action |
|-----------|---------|--------|
| `products.product_category_name` (610) and the 3 length/photo columns that go missing with it | Data-quality gap — these products still have real sales | Fill category with an explicit `unknown` so they are **counted, not dropped**; leave numeric descriptors as NaN (never fabricate measurements) |
| `products.product_weight_g / length / height / width` (2) | Negligible | Leave as NaN |
| `orders.order_approved_at` (160) | Mostly cancelled orders (141) — legitimate | Leave as NaT |
| `orders.order_delivered_carrier_date / customer_date` | Legitimate for orders not yet delivered **except** 8 orders marked `delivered` with no delivery date | Flag those 8 as `delivery_date_missing` and exclude them from delivery-time averages |
| `reviews.review_comment_title` (~88%) / `review_comment_message` (~59%) | Legitimate — most customers only leave a star rating | Fill with empty string and add a `has_comment` boolean so analysts don't have to remember |
| `categories` | 2 Portuguese categories present in `products` but absent from the translation table | Add `pc_gamer` → `pc_gamer`, `portateis_cozinha_e_preparadores_de_alimentos` → `kitchen_portables_food_preparers` |

### What I did

In [5]:
# 2a. Products: explicit 'unknown' category instead of NaN
p = clean["products"]
p["product_category_name"] = p["product_category_name"].fillna("unknown")

# 2b. Category translation: add the two missing rows, plus 'unknown'
cat = clean["categories"]
missing = pd.DataFrame({
    "product_category_name":         ["pc_gamer", "portateis_cozinha_e_preparadores_de_alimentos", "unknown"],
    "product_category_name_english": ["pc_gamer", "kitchen_portables_food_preparers", "unknown"],
})
clean["categories"] = pd.concat([cat, missing], ignore_index=True).drop_duplicates("product_category_name")
untranslated = set(p.product_category_name) - set(clean["categories"].product_category_name)
assert not untranslated, untranslated

# 2c. Orders: flag impossible 'delivered but no date' records (dates are parsed in section 3)
o = clean["orders"]
o["delivery_date_missing"] = (o.order_status == "delivered") & o.order_delivered_customer_date.isna()

# 2d. Reviews: empty string + explicit flag
r = clean["reviews"]
r["has_comment"] = r.review_comment_message.notna()
r[["review_comment_title", "review_comment_message"]] = r[["review_comment_title", "review_comment_message"]].fillna("")

print("products with category 'unknown':", int((p.product_category_name == "unknown").sum()))
print("delivered orders flagged with missing delivery date:", int(o.delivery_date_missing.sum()))
print("reviews with a written comment:", int(r.has_comment.sum()), "of", len(r))

products with category 'unknown': 610
delivered orders flagged with missing delivery date: 8
reviews with a written comment: 40775 of 98673


### What the client would see
No product revenue disappears from the dashboard because of a missing label — those 610 products show up under an honest `unknown` bucket. Delivery-time KPIs are not skewed by 8 broken records, and every category has an English name on the charts.

## 3. Problem: Date types

### What was wrong
All eight timestamp columns were loaded as plain text (`str`). You cannot compute delivery time, group by month, or filter a date range on text, and any BI tool would treat them as labels rather than a timeline.

In [6]:
date_cols = {
    "orders":  ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
                "order_delivered_customer_date", "order_estimated_delivery_date"],
    "items":   ["shipping_limit_date"],
    "reviews": ["review_creation_date", "review_answer_timestamp"],
}
pd.DataFrame([(t, c, str(clean[t][c].dtype)) for t, cols in date_cols.items() for c in cols],
             columns=["table", "column", "dtype_before"])

,table,column,dtype_before
0,orders,order_purchase_timestamp,str
1,orders,order_approved_at,str
2,orders,order_delivered_carrier_date,str
3,orders,order_delivered_customer_date,str
4,orders,order_estimated_delivery_date,str
5,items,shipping_limit_date,str
6,reviews,review_creation_date,str
7,reviews,review_answer_timestamp,str


### What I did

In [7]:
for t, cols in date_cols.items():
    for c in cols:
        clean[t][c] = pd.to_datetime(clean[t][c], format="%Y-%m-%d %H:%M:%S", errors="raise")

o = clean["orders"]
# Derived fields the dashboard needs
o["purchase_date"]       = o.order_purchase_timestamp.dt.normalize()
o["purchase_month"]      = o.order_purchase_timestamp.dt.to_period("M").dt.to_timestamp()
o["delivery_days"]       = (o.order_delivered_customer_date - o.order_purchase_timestamp).dt.total_seconds() / 86400
o["delivery_delay_days"] = (o.order_delivered_customer_date - o.order_estimated_delivery_date).dt.total_seconds() / 86400
o["delivered_on_time"]   = np.where(o.order_delivered_customer_date.notna(), o.delivery_delay_days <= 0, np.nan)
o["delivery_days"] = o.delivery_days.round(1)
o["delivery_delay_days"] = o.delivery_delay_days.round(1)

# Sanity checks a client would want to hear about
assert (o.order_delivered_customer_date.dropna() >= o.order_purchase_timestamp[o.order_delivered_customer_date.notna()]).all(), "delivered before purchase"
print("Purchase window:", o.order_purchase_timestamp.min().date(), "->", o.order_purchase_timestamp.max().date())
print("Mean delivery days (delivered orders):", round(o.loc[o.order_status == "delivered", "delivery_days"].mean(), 1))
print("On-time delivery rate:", f"{o.delivered_on_time.dropna().mean():.1%}")
o[["order_purchase_timestamp", "purchase_month", "order_delivered_customer_date", "delivery_days", "delivery_delay_days", "delivered_on_time"]].head()

Purchase window: 2016-09-04 -> 2018-10-17
Mean delivery days (delivered orders): 12.6
On-time delivery rate: 91.9%


,order_purchase_timestamp,purchase_month,order_delivered_customer_date,delivery_days,delivery_delay_days,delivered_on_time
0,2017-10-02 10:56:33,2017-10-01,2017-10-10 21:25:13,8.4,-7.1,1.0
1,2018-07-24 20:41:37,2018-07-01,2018-08-07 15:27:45,13.8,-5.4,1.0
2,2018-08-08 08:38:49,2018-08-01,2018-08-17 18:06:29,9.4,-17.2,1.0
3,2017-11-18 19:28:06,2017-11-01,2017-12-02 00:28:42,13.2,-13.0,1.0
4,2018-02-13 21:18:39,2018-02-01,2018-02-16 18:17:02,2.9,-9.2,1.0


### What the client would see
A real time axis. Monthly revenue and order counts group correctly, the *average delivery days* tile is a true calculation (purchase → delivered), and late deliveries can be filtered with a single boolean.

> Note the dataset's edges: September–December 2016 has only a handful of orders (Olist was onboarding), and September–October 2018 is a partial month. Both are kept in the data but should be read with that in mind on the trend chart.

## 4. Problem: Currency

### What was wrong
1. `price` and `freight_value` are plain floats with no unit. The dataset is Brazilian, so these are **BRL (R$)**, but nothing in the file says so — a dashboard viewer outside Brazil would misread them.
2. `payment_value` is recorded per payment, and an order can have several payments (vouchers + card). When I reconcile *sum of payments* against *sum of item prices + freight* per order, **303 orders do not match**, and 1 order has items but no payment row at all.

In [8]:
it, pay = clean["items"], clean["payments"]
recon = (it.groupby("order_id")[["price", "freight_value"]].sum().sum(axis=1).rename("items_total")
           .to_frame()
           .join(pay.groupby("order_id").payment_value.sum().rename("payments_total"), how="left"))
recon["difference"] = (recon.payments_total - recon.items_total).round(2)
mismatch = recon[(recon.difference.abs() > 0.01) | recon.payments_total.isna()]
print(f"Orders reconciled: {len(recon):,}   mismatched: {len(mismatch):,}   no payment row: {int(recon.payments_total.isna().sum())}")
mismatch.difference.describe().round(2)

Orders reconciled: 98,666   mismatched: 304   no payment row: 1


count    303.00
mean       9.48
std       18.08
min      -51.62
25%        1.36
50%        5.41
75%       11.83
max      182.81
Name: difference, dtype: float64

### What I did
* Renamed the money columns with an explicit `_brl` suffix and rounded to 2 decimal places (cents).
* Added a `_usd` column for each. Olist spans 2016–2018, during which the real traded between roughly 3.1 and 4.2 per US dollar. Rather than silently apply a daily rate the client has not approved, I used a **single documented reference rate of 1 USD = 3.65 BRL** (approximate 2017–2018 average) and exposed it as a constant so it can be changed in one place.
* Revenue on the dashboard is defined as **item price + freight for orders that were not cancelled or unavailable** — i.e. what the customer was actually billed for goods and shipping. Payments are kept as a separate table.
* Flagged the 303 mismatched orders plus the 1 order with no payment row (304 in total) with `payment_mismatch=True` instead of altering either side; the differences are small (median R$5, max R$183) and mostly voucher-related, so they are documented, not "fixed".

In [9]:
USD_PER_BRL = 1 / 3.65   # documented reference rate; change here if the client prefers another

it = clean["items"].rename(columns={"price": "price_brl", "freight_value": "freight_brl"})
it["price_brl"]   = it.price_brl.round(2)
it["freight_brl"] = it.freight_brl.round(2)
it["price_usd"]   = (it.price_brl * USD_PER_BRL).round(2)
it["freight_usd"] = (it.freight_brl * USD_PER_BRL).round(2)
clean["items"] = it

pay = clean["payments"].rename(columns={"payment_value": "payment_value_brl"})
pay["payment_value_brl"] = pay.payment_value_brl.round(2)
pay["payment_value_usd"] = (pay.payment_value_brl * USD_PER_BRL).round(2)
clean["payments"] = pay

o = clean["orders"]
o["payment_mismatch"] = o.order_id.isin(mismatch.index)
print("orders flagged payment_mismatch:", int(o.payment_mismatch.sum()))
it.head(3)

orders flagged payment_mismatch: 304


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price_brl,freight_brl,price_usd,freight_usd
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29,16.14,3.64
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93,65.73,5.46
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87,54.52,4.90


### What the client would see
Every money column on the dashboard is unambiguous (R$ or US$), totals reconcile to the cent, and the handful of orders where billing and payment disagree are visible as a filter rather than hidden inside an average.

## 5. Small consistency fixes (bundled)

Not large enough to be their own section, but worth recording:
* City names in `geolocation` mix accented and unaccented spellings (`são paulo` vs `sao paulo`). Normalised to lowercase ASCII so joins on city work.
* State codes upper-cased and stripped.
* Added a `region` column (North / Northeast / Centre-West / Southeast / South) to customers and sellers using the official IBGE grouping, so the dashboard can roll 27 states up to 5 regions.

In [10]:
def ascii_lower(s: pd.Series) -> pd.Series:
    return (s.astype(str).str.strip().str.lower()
             .map(lambda x: unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode()))

g = clean["geolocation"]
g["geolocation_city"] = ascii_lower(g.geolocation_city)
g["geolocation_state"] = g.geolocation_state.str.upper().str.strip()
clean["geolocation"] = g.drop_duplicates().reset_index(drop=True)

REGION = {
  "AC":"North","AM":"North","AP":"North","PA":"North","RO":"North","RR":"North","TO":"North",
  "AL":"Northeast","BA":"Northeast","CE":"Northeast","MA":"Northeast","PB":"Northeast","PE":"Northeast","PI":"Northeast","RN":"Northeast","SE":"Northeast",
  "DF":"Centre-West","GO":"Centre-West","MS":"Centre-West","MT":"Centre-West",
  "ES":"Southeast","MG":"Southeast","RJ":"Southeast","SP":"Southeast",
  "PR":"South","RS":"South","SC":"South",
}
for t, col in [("customers", "customer_state"), ("sellers", "seller_state")]:
    df = clean[t]
    df[col] = df[col].str.upper().str.strip()
    df[t[:-1] + "_region"] = df[col].map(REGION)
    assert df[t[:-1] + "_region"].notna().all(), f"unmapped state in {t}"
    df[t[:-1] + "_city"] = ascii_lower(df[t[:-1] + "_city"])

print("geolocation rows after city normalisation + re-dedupe:", f"{len(clean['geolocation']):,}")
clean["customers"].customer_region.value_counts()

geolocation rows after city normalisation + re-dedupe: 720,495


customer_region
Southeast      68266
South          14148
Northeast       9394
Centre-West     5782
North           1851
Name: count, dtype: int64

## 6. Build the dashboard tables

Two flat tables feed Looker Studio:

* **`orders_fact`** — one row per order, with revenue, customer state/region, purchase month and delivery days. Looker Studio aggregates this itself for the revenue trend, orders-by-month, revenue-by-region and average-delivery-days tiles.
* **`top_products`** — the ten products with the highest revenue, with an English category label (Olist product IDs are anonymised hashes, so each product is labelled `category · first 6 chars of id`).

In [11]:
o, it, cu, p, cat = clean["orders"], clean["items"], clean["customers"], clean["products"], clean["categories"]

VALID_STATUS = ~o.order_status.isin(["canceled", "unavailable"])

order_rev = (it.groupby("order_id")
               .agg(items=("order_item_id", "count"),
                    revenue_brl=("price_brl", "sum"), freight_brl=("freight_brl", "sum"))
               .reset_index())
order_rev["revenue_brl"] = (order_rev.revenue_brl + order_rev.freight_brl).round(2)   # goods + shipping billed
order_rev["revenue_usd"] = (order_rev.revenue_brl * USD_PER_BRL).round(2)

orders_fact = (o[VALID_STATUS]
               .merge(order_rev, on="order_id", how="inner")
               .merge(cu[["customer_id", "customer_state", "customer_region"]], on="customer_id", how="left"))
orders_fact["delivery_days"] = orders_fact.delivery_days.where(orders_fact.order_status == "delivered")
orders_fact = orders_fact[["order_id", "order_status", "purchase_date", "purchase_month", "customer_state", "customer_region",
                           "items", "revenue_brl", "revenue_usd", "delivery_days", "delivered_on_time", "payment_mismatch"]]
orders_fact["purchase_date"]  = orders_fact.purchase_date.dt.strftime("%Y-%m-%d")
orders_fact["purchase_month"] = orders_fact.purchase_month.dt.strftime("%Y-%m")

prod = (it.merge(p[["product_id", "product_category_name"]], on="product_id", how="left")
          .merge(cat, on="product_category_name", how="left")
          .merge(o.loc[VALID_STATUS, ["order_id"]], on="order_id", how="inner"))
top_products = (prod.groupby(["product_id", "product_category_name_english"])
                    .agg(units_sold=("order_item_id", "count"), revenue_brl=("price_brl", "sum"))
                    .reset_index().sort_values("revenue_brl", ascending=False).head(10))
top_products["revenue_brl"] = top_products.revenue_brl.round(2)
top_products["revenue_usd"] = (top_products.revenue_brl * USD_PER_BRL).round(2)
top_products["product_label"] = top_products.product_category_name_english + " · " + top_products.product_id.str[:6]
top_products = top_products[["product_label", "product_category_name_english", "product_id", "units_sold", "revenue_brl", "revenue_usd"]].reset_index(drop=True)

orders_fact.to_csv(DASH / "orders_fact.csv", index=False)
top_products.to_csv(DASH / "top_products.csv", index=False)

print(f"orders_fact: {len(orders_fact):,} orders   total revenue R$ {orders_fact.revenue_brl.sum():,.2f}")
top_products

orders_fact: 98,199 orders   total revenue R$ 15,735,527.03


,product_label,product_category_name_english,product_id,units_sold,revenue_brl,revenue_usd
0,health_beauty · bb50f2,health_beauty,bb50f2e236e5eea0100680137654686c,195,63885.00,17502.74
1,health_beauty · 6cdd53,health_beauty,6cdd53843498f92890544667809f1595,156,54730.20,14994.58
2,computers · d6160f,computers,d6160fb7873f184099d9bc95e30376af,35,48899.34,13397.08
3,computers_accessories · d1c427,computers_accessories,d1c427060a0f73f6b889a5c7c61f2ac4,341,46916.51,12853.84
4,bed_bath_table · 99a478,bed_bath_table,99a4788cb24856965c36a24e339b6058,487,42938.66,11764.02
5,computers_accessories · 3dd2a1,computers_accessories,3dd2a17168ec895c781a9191c1e95ad7,274,41082.60,11255.51
6,baby · 25c385,baby,25c38557cf793876c5abdd5931f922db,38,38907.32,10659.54
7,cool_stuff · 5f504b,cool_stuff,5f504b3a1c75b73d6151be81eb05bdc9,63,37733.90,10338.05
8,watches_gifts · 53b36d,watches_gifts,53b36df67ebb7c41585e8d54d6772e08,323,37683.42,10324.22
9,furniture_decor · aca2eb,furniture_decor,aca2eb7d00ea1a7b8ebd4e68314663af,527,37608.90,10303.81


Headline numbers the dashboard should reproduce:

In [12]:
monthly = (orders_fact.groupby("purchase_month")
             .agg(orders=("order_id", "count"), revenue_brl=("revenue_brl", "sum"), avg_delivery_days=("delivery_days", "mean"))
             .round(1))
by_region = orders_fact.groupby("customer_region").revenue_brl.sum().sort_values(ascending=False).round(2)
print("Average delivery days (all delivered orders):", round(orders_fact.delivery_days.mean(), 1))
print("\nRevenue by region (R$):"); print(by_region.to_string())
monthly

Average delivery days (all delivered orders): 12.6

Revenue by region (R$):
customer_region
Southeast      10159955.08
South           2279510.90
Northeast       1872583.31
Centre-West     1013752.38
North            409725.36


,orders,revenue_brl,avg_delivery_days
purchase_month,,,
2016-09,2,279.7,54.8
2016-10,290,51354.5,19.6
2016-12,1,19.6,4.7
2017-01,787,136943.5,12.6
2017-02,1718,283561.7,13.2
2017-03,2617,425618.0,13.0
2017-04,2377,405848.6,14.9
2017-05,3640,582710.8,11.3
2017-06,3205,499652.2,12.0


## 7. Export clean tables and before/after summary

In [13]:
for name, df in clean.items():
    df.to_csv(CLEAN / f"{name}.csv", index=False)

summary = pd.DataFrame({
    "rows_before": before_rows,
    "rows_after": {k: len(v) for k, v in clean.items()},
    "nulls_before": {k: int(v.isna().sum().sum()) for k, v in raw.items()},
    # nulls counted on the ORIGINAL columns only (derived columns like delivery_days are legitimately NaN for undelivered orders)
    "nulls_after": {k: int(v[[c for c in v.columns if c in raw[k].columns]].isna().sum().sum()) for k, v in clean.items()},
    "text_date_cols_before": {k: len(date_cols.get(k, [])) for k in clean},
    "text_date_cols_after": 0,
})
summary["rows_removed"] = summary.rows_before - summary.rows_after
summary

,rows_before,rows_after,nulls_before,nulls_after,text_date_cols_before,text_date_cols_after,rows_removed
orders,99441,99441,4908,4908,5,0,0
items,112650,112650,0,0,1,0,0
payments,103886,103886,0,0,0,0,0
reviews,99224,98673,145903,0,2,0,551
customers,99441,99441,0,0,0,0,0
products,32951,32951,2448,1838,0,0,0
sellers,3095,3095,0,0,0,0,0
geolocation,1000163,720495,0,0,0,0,279668
categories,71,74,0,0,0,0,-3


Remaining nulls after cleaning are all **intentional**: order dates for orders that were never approved/shipped/delivered, and a handful of physical product dimensions that were never recorded. None of them affect a dashboard metric.

## Files produced

```
data/clean/        orders, items, payments, reviews, customers, products, sellers, geolocation, categories  (.csv)
data/dashboard/    orders_fact.csv  (one row per billable order)
                   top_products.csv (top 10 products by revenue)
```